In [8]:
import os
import firebase_admin
from firebase_admin import credentials, firestore

if not firebase_admin._apps:
    cred = credentials.Certificate(os.environ["GOOGLE_APPLICATION_CREDENTIALS"])
    firebase_admin.initialize_app(cred)

db = firestore.client()

In [ ]:
import json
import requests

api_endpoint = "https://foster-ulcer-ai-backend-429230748709.asia-southeast3.run.app/case_detail"

payload = {
    "nrc_id": "8757446557345",
    "patient_name": "TEST GCR (call from local python)",
    "phone_no": "01457864357",
    "dob": "10/04/2001",
    "gender": "female",
    "height_cm": "160",
    "weight_kg": "120",
    "medical_history": "Allergy",
    "diabetes": {
        "has_diabetes": "Yes",
        "years": "5-10y",
        "risk_history": ["Past Ulcer"],
        "complications": ["Heart"]
    },
    "created_at": "2026-03-09 10:08:35"
}

response = requests.post(
    api_endpoint,
    data={"patient_data": json.dumps(payload)}
)

print(response.status_code, response.text)


200 {"status":"success","patient_id":"PT-2603-00004","photo_url":null,"message":"Profile created for TEST GCR (call from local python)"}


In [9]:
import json
import requests

api_endpoint = "https://foster-ulcer-ai-backend-429230748709.asia-southeast3.run.app/case_detail"
payload = {
  "case_id":"CS-260313-00006"
}

response = requests.post(
    api_endpoint,
    data=json.dumps(payload),
    headers={"Content-Type": "application/json"}
)


print(response.status_code, response.text)


200 {"status":"success","case":{"status":"DOCTOR_REVIEW","current_analysis_id":"AN-20260313093659","case_id":"CS-260313-00006","current_timestamps":{"appointment_at":null,"doctor_review_at":null,"treatment_active_at":null,"plan_issued_at":null,"created_at":"2026-03-13T09:36:59.617000+00:00","updated_at":"2026-03-13T09:36:59.617000+00:00","completed_at":null,"analyze_at":"2026-03-13T09:36:59.617000+00:00"},"created_by_nurse":null,"current_image":{"image_folder_url":"https://storage.googleapis.com/foster-ulcer-ai.firebasestorage.app/cases/CS-260313-00006/REC-00001/CS-260313-00006-REC-00001-20260313093529.jpg"},"current_vital_signs":{"temperature":"High","heart_rate":"High","blood_glucose":"High","blood_pressure":"High","respiratory_rate":"Normal"},"current_analysis":{"confidence":0.65,"red_flag":false,"description":"The patient presents with a diabetic foot ulcer on the forefoot with necrosis. There is evidence of infection, but deep structure involvement is unknown. Ischemia status is a

In [ ]:
import argparse
import copy
import json
import mimetypes
import os
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional

import requests


DEFAULT_CREATE_CASE_PAYLOAD = {
    "status": "CREATION",
    "vitals": {
        "temperature": "High",
        "blood_pressure": "High",
        "heart_rate": "High",
        "respiratory_rate": "High",
        "blood_sugar": "High",
    },
    "meta": {
        "sent_at": None,
    },
}

DEFAULT_SEND_TO_DOCTOR_TEMPLATE = {
    "status": "DOCTOR_REVIEW",
    "urgency": "URGENT",
    "vital_signs": {
        "temperature": "High",
        "blood_pressure": "High",
        "blood_glucose": "High",
        "heart_rate": "High",
        "respiratory_rate": "High",
    },
    "wound_detail": {
        "location_primary": "dorsal_aspect",
        "location_detail": "foot",
        "wound_type": "ulcer",
        "shape": "irregular",
        "size": {"width_cm": 3.0, "length_cm": 4.0},
        "depth_category": "full_thickness",
        "bed": {"slough_pct": 0, "necrotic_pct": 0},
        "edge_description": "irregular",
        "periwound_status": "erythematous",
        "discharge": {
            "volume": "minimal",
            "type": "serosanguineous (pink)",
        },
        "odor_presence": "faint",
        "pain_score": 8,
        "has_infection": True,
        "skin_condition": "macerated",
    },
    "ischemia": {
        "points": [0, 1],
        "pulse": "yes",
        "checklist": [
            "Black tissue (tissue loss)",
            "Cold foot",
            "Color (pale/blue)",
        ],
    },
    "infection": {
        "checklist": [
            "Pain (tender or hurts to touch)",
            "Pus / Goo (Purulent discharge)",
            "Swelling (puffy, tight, hard)",
            "Warmth (hotter than other foot)",
        ],
        "erythema_extent": "gt_0_5_cm",
        "probe_to_bone_test": "negative",
        "has_deep_abscess_or_fasciitis": "false",
    },
    "neuropathy": {
        "points": [0, 1, 2, 3, 4, 5, 6, 7, 8],
    },
    "sinbad": {
        "site": "Forefoot",
        "ischemia": "Yes",
        "neuropathy": "Yes",
        "infection": "Yes",
        "area": ">= 1 cm²",
        "depth": "Skin only",
    },
    "lab_results": {
        "wbc_count": "8000",
        "crp": "10",
        "esr": "30",
        "procalcitonin": "1",
    },
    "vascular": {
        "abi_value": "0",
        "ankle_pressure_mmHg": "0",
        "toe_pressure_mmHg": "0",
        "tcpo2_mmHg": "0",
    },
    "gangrene_extent": "heel_full_thickness",
}


def pretty(data: Any) -> str:
    return json.dumps(data, ensure_ascii=False, indent=2, default=str)


class APITestClient:
    def __init__(self, base_url: str, timeout: int = 120):
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session()
        self.timeout = timeout

    def _url(self, path: str) -> str:
        return f"{self.base_url}{path}"

    def post_json(self, path: str, payload: Dict[str, Any]) -> Dict[str, Any]:
        url = self._url(path)
        print(f"\n[REQUEST] POST {path} (json)")
        print(pretty(payload))
        response = self.session.post(url, json=payload, timeout=self.timeout)
        self._raise_for_status(response)
        data = self._safe_json(response)
        print(f"[RESPONSE] {response.status_code}")
        print(pretty(data))
        return data

    def post_multipart(
        self,
        path: str,
        data: Dict[str, Any],
        file_field_name: str,
        file_path: str,
    ) -> Dict[str, Any]:
        url = self._url(path)
        filename = os.path.basename(file_path)
        mime = mimetypes.guess_type(filename)[0] or "application/octet-stream"
        print(f"\n[REQUEST] POST {path} (multipart)")
        print(pretty({"form": data, "file_field_name": file_field_name, "file": filename}))
        with open(file_path, "rb") as f:
            files = {file_field_name: (filename, f, mime)}
            response = self.session.post(url, data=data, files=files, timeout=self.timeout)
        self._raise_for_status(response)
        data = self._safe_json(response)
        print(f"[RESPONSE] {response.status_code}")
        print(pretty(data))
        return data

    @staticmethod
    def _safe_json(response: requests.Response) -> Any:
        try:
            return response.json()
        except ValueError:
            return {"raw_text": response.text}

    @staticmethod
    def _raise_for_status(response: requests.Response) -> None:
        try:
            response.raise_for_status()
        except requests.HTTPError as exc:
            print("[ERROR BODY]")
            print(response.text)
            raise exc


def load_json_file(path: Optional[str]) -> Optional[Dict[str, Any]]:
    if not path:
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def create_patient_if_needed(
    client: APITestClient,
    patient_id: Optional[str],
    profile_payload_path: Optional[str],
    profile_photo_path: Optional[str],
) -> str:
    if patient_id:
        print(f"\n[INFO] Using existing patient_id: {patient_id}")
        return patient_id

    payload = load_json_file(profile_payload_path) or {
        "first_name": "Test",
        "last_name": "Patient",
        "purpose": "Create case",
        "gender": "male",
        "age": 60,
    }

    if not profile_photo_path:
        raise ValueError(
            "You must provide either --patient-id or both --profile-payload and --profile-photo."
        )

    response = client.post_multipart(
        "/create-patient-profile",
        data=payload,
        file_field_name="profile_photo",
        file_path=profile_photo_path,
    )
    patient_id = response.get("patient_id")
    if not patient_id:
        raise RuntimeError("/create-patient-profile succeeded but patient_id was missing.")
    return patient_id


def main() -> None:
    parser = argparse.ArgumentParser(description="Smoke test for Foster Ulcer AI API flow")
    parser.add_argument("--base-url", default="http://127.0.0.1:8000", help="API base URL")
    parser.add_argument("--patient-id", help="Use an existing patient ID and skip patient creation")
    parser.add_argument("--profile-payload", help="JSON file for /create-patient-profile form fields")
    parser.add_argument("--profile-photo", help="Profile photo path for /create-patient-profile")
    parser.add_argument("--wound-image", required=True, help="Image path for analyze endpoints")
    parser.add_argument("--create-case-payload", help="Override JSON for /create-case")
    parser.add_argument("--send-to-doctor-payload", help="Override JSON for /send-to-doctor")
    parser.add_argument(
        "--output-dir",
        default="api_test_output",
        help="Directory to save step-by-step responses",
    )
    parser.add_argument(
        "--analyze-file-field",
        default="file",
        help="Multipart field name for image upload endpoints",
    )
    args = parser.parse_args()

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    client = APITestClient(base_url=args.base_url)

    patient_id = create_patient_if_needed(
        client=client,
        patient_id=args.patient_id,
        profile_payload_path=args.profile_payload,
        profile_photo_path=args.profile_photo,
    )

    create_case_payload = load_json_file(args.create_case_payload) or copy.deepcopy(DEFAULT_CREATE_CASE_PAYLOAD)
    create_case_payload["patient_id"] = patient_id
    create_case_payload.setdefault("meta", {})["sent_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    create_case_response = client.post_json("/create-case", create_case_payload)
    case_id = create_case_response["case_id"]
    record_id = create_case_response["record_id"]

    analyze_fillin_response = client.post_multipart(
        "/analyze-fillin",
        data={"case_id": case_id, "record_id": record_id},
        file_field_name=args.analyze_file_field,
        file_path=args.wound_image,
    )

    analyze_wound_response = client.post_multipart(
        "/analyze-wound",
        data={"case_id": case_id, "record_id": record_id},
        file_field_name=args.analyze_file_field,
        file_path=args.wound_image,
    )

    send_to_doctor_payload = load_json_file(args.send_to_doctor_payload) or copy.deepcopy(DEFAULT_SEND_TO_DOCTOR_TEMPLATE)
    send_to_doctor_payload["patient_id"] = patient_id
    send_to_doctor_payload["case_id"] = case_id
    send_to_doctor_payload["record_id"] = record_id

    # Optionally blend in analyze-fillin result to reduce repeated manual typing.
    fillin_analysis = analyze_fillin_response.get("analysis") or {}
    if fillin_analysis:
        send_to_doctor_payload.setdefault("wound_detail", {})
        send_to_doctor_payload["wound_detail"].update(
            {
                "location_primary": fillin_analysis.get("location_primary", send_to_doctor_payload["wound_detail"].get("location_primary")),
                "location_detail": fillin_analysis.get("location_detail", send_to_doctor_payload["wound_detail"].get("location_detail")),
                "wound_type": fillin_analysis.get("wound_type", send_to_doctor_payload["wound_detail"].get("wound_type")),
                "shape": fillin_analysis.get("shape", send_to_doctor_payload["wound_detail"].get("shape")),
                "depth_category": fillin_analysis.get("depth_category", send_to_doctor_payload["wound_detail"].get("depth_category")),
                "edge_description": fillin_analysis.get("edge_description", send_to_doctor_payload["wound_detail"].get("edge_description")),
                "periwound_status": fillin_analysis.get("periwound_status", send_to_doctor_payload["wound_detail"].get("periwound_status")),
                "odor_presence": fillin_analysis.get("odor_presence", send_to_doctor_payload["wound_detail"].get("odor_presence")),
                "pain_score": fillin_analysis.get("pain_score", send_to_doctor_payload["wound_detail"].get("pain_score")),
                "has_infection": fillin_analysis.get("has_infection", send_to_doctor_payload["wound_detail"].get("has_infection")),
                "skin_condition": fillin_analysis.get("skin_condition", send_to_doctor_payload["wound_detail"].get("skin_condition")),
                "size": {
                    "width_cm": fillin_analysis.get("size_width_cm", send_to_doctor_payload["wound_detail"].get("size", {}).get("width_cm")),
                    "length_cm": fillin_analysis.get("size_length_cm", send_to_doctor_payload["wound_detail"].get("size", {}).get("length_cm")),
                },
                "bed": {
                    "slough_pct": fillin_analysis.get("bed_slough_pct", send_to_doctor_payload["wound_detail"].get("bed", {}).get("slough_pct")),
                    "necrotic_pct": fillin_analysis.get("bed_necrotic_pct", send_to_doctor_payload["wound_detail"].get("bed", {}).get("necrotic_pct")),
                },
                "discharge": {
                    "volume": fillin_analysis.get("discharge_volume", send_to_doctor_payload["wound_detail"].get("discharge", {}).get("volume")),
                    "type": fillin_analysis.get("discharge_type", send_to_doctor_payload["wound_detail"].get("discharge", {}).get("type")),
                },
            }
        )

    wound_analysis_raw = analyze_wound_response.get("analysis")
    if isinstance(wound_analysis_raw, str):
        try:
            wound_analysis_parsed = json.loads(wound_analysis_raw)
        except json.JSONDecodeError:
            wound_analysis_parsed = {"raw_analysis": wound_analysis_raw}
    else:
        wound_analysis_parsed = wound_analysis_raw or {}

    if isinstance(wound_analysis_parsed, dict):
        send_to_doctor_payload["analysis"] = wound_analysis_parsed.get("AI_analysis")
        send_to_doctor_payload["treatment_plan"] = wound_analysis_parsed.get("treatment_plan")
        plan_tasks = (wound_analysis_parsed.get("treatment_plan") or {}).get("plan_tasks")
        if plan_tasks:
            send_to_doctor_payload["task_list"] = plan_tasks

    send_to_doctor_response = client.post_json("/send-to-doctor", send_to_doctor_payload)

    artifacts = {
        "01_create_case_response.json": create_case_response,
        "02_analyze_fillin_response.json": analyze_fillin_response,
        "03_analyze_wound_response.json": analyze_wound_response,
        "04_send_to_doctor_payload.json": send_to_doctor_payload,
        "05_send_to_doctor_response.json": send_to_doctor_response,
        "summary.json": {
            "patient_id": patient_id,
            "case_id": case_id,
            "record_id": record_id,
            "analysis_id": send_to_doctor_response.get("analysis_id"),
            "plan_id": send_to_doctor_response.get("plan_id"),
        },
    }

    for filename, payload in artifacts.items():
        (output_dir / filename).write_text(pretty(payload), encoding="utf-8")

    print("\n[DONE] Flow completed successfully.")
    print(pretty(artifacts["summary.json"]))
    print(f"Saved output files to: {output_dir.resolve()}")


if __name__ == "__main__":
    main()
